# Notebook 2 — Detailed Analysis

**Credit Risk EDA Case Study** | Sankalp Seksaria & Vaibhav Parakh

This notebook covers:
- Univariate analysis (categorical & numerical)
- Bivariate analysis (categorical & numerical vs TARGET)
- Multivariate / correlation analysis
- Merged dataset analysis (current + previous applications)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from src.eda_utils import (
    load_application_data,
    load_previous_application,
    plot_distribution,
    plot_default_analysis,
    plot_correlation_heatmap,
    plot_count_by_target,
    plot_violin,
    default_rate_by_category,
    high_correlation_pairs,
    cap_outliers,
    chi_square_test,
)
from src.config import DATA_PROCESSED_PATH, TARGET_COLUMN


In [ ]:
# Load cleaned data (produced by Notebook 1)
curr_appl = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "application_cleaned.csv"))
prev_appl = load_previous_application()
print(f"Current application: {curr_appl.shape}")
print(f"Previous application: {prev_appl.shape}")


## Step 4A — Target Variable Distribution


In [ ]:
target_counts = curr_appl[TARGET_COLUMN].value_counts()
print(target_counts)

fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind="bar", color=["#2ecc71", "#e74c3c"], ax=ax)
ax.set_title("Target Variable Distribution")
ax.set_xticklabels(["No Difficulty (0)", "Payment Difficulty (1)"], rotation=0)
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


### Observation
There is a significant class imbalance — applicants with payment difficulties (TARGET=1) represent a small minority.  
For a fair comparison we split the dataset by target value before analysis.


In [ ]:
# Separate sub-datasets for easier comparison
target_0 = curr_appl[curr_appl[TARGET_COLUMN] == 0]
target_1 = curr_appl[curr_appl[TARGET_COLUMN] == 1]
print(f"Target 0 (no difficulty): {len(target_0):,}  |  Target 1 (difficulty): {len(target_1):,}")


## Step 4B — Univariate Analysis: Categorical Variables


In [ ]:
cat_features = ["NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE",
                "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE"]

for feat in cat_features:
    if feat in curr_appl.columns:
        fig = plot_count_by_target(curr_appl, feat)
        plt.show()


## Step 4C — Bivariate Analysis: Categorical vs Default Rate


In [ ]:
for feat in cat_features:
    if feat in curr_appl.columns:
        rate = default_rate_by_category(curr_appl, feat)
        print(f"
--- Default rate by {feat} ---")
        print(rate.to_string())
        fig = plot_default_analysis(curr_appl, feat)
        plt.show()


## Step 4D — Univariate Analysis: Numerical Variables


In [ ]:
num_features = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
                "YEARS_BIRTH", "YEARS_EMPLOYED", "CNT_CHILDREN"]

for feat in num_features:
    if feat in curr_appl.columns:
        fig = plot_distribution(curr_appl, feat)
        plt.show()


## Step 4E — Bivariate Analysis: Numerical vs TARGET (Violin Plots)


In [ ]:
for feat in num_features:
    if feat in curr_appl.columns:
        fig = plot_violin(curr_appl, x=TARGET_COLUMN, y=feat)
        plt.show()


## Step 4F — Multivariate: Correlation Analysis


In [ ]:
fig = plot_correlation_heatmap(curr_appl)
plt.show()

# Identify highly correlated feature pairs
high_corr = high_correlation_pairs(curr_appl)
print("
Highly correlated feature pairs:")
high_corr.head(10)


## Step 5 — Merge with Previous Application


In [ ]:
merged = curr_appl.merge(prev_appl, on="SK_ID_CURR", how="left", suffixes=("", "_prev"))
print(f"Merged dataset: {merged.shape}")


## Step 6 — Analysis of Merged Dataset (Categorical)


In [ ]:
prev_cat_features = ["NAME_CONTRACT_STATUS", "NAME_YIELD_GROUP", "PRODUCT_COMBINATION"]

for feat in prev_cat_features:
    if feat in merged.columns:
        rate = default_rate_by_category(merged, feat)
        print(f"
--- Default rate by {feat} ---")
        print(rate.head(10).to_string())
        fig = plot_default_analysis(merged, feat)
        plt.show()


## Summary

Key analytical findings from this notebook are documented in .  
Proceed to **Notebook 3** for final business insights and recommendations.
